# Les interfaces

En POO, une interface définit un **contrat** : elle spécifie les méthodes qu'une classe **doit** implémenter, sans fournir d'implémentation.

### Objectifs
- Garantir une **structure commune** entre les classes
- Favoriser la **modularité** en travaillant avec des abstractions
- Permettre le **découplage** entre le code appelant et les implémentations

---

## Le Duck Typing

En Python, le typage repose sur le **Duck Typing** :

> *"Si ça marche comme un canard et que ça cancane comme un canard, alors c'est un canard."*

Un objet est considéré compatible avec un type s'il **implémente les méthodes attendues**, indépendamment de son héritage. Pas besoin de déclarer explicitement qu'il implémente une interface.

In [ ]:
# Deux classes SANS AUCUN lien d'héritage entre elles...
class Canard:
    def crier(self):
        return "Coin coin !"

class Robot:
    def crier(self):
        return "Bip bip !"


# ...et une fonction qui accepte les deux : elle ne regarde pas le TYPE
# de l'objet, seulement s'il possède la méthode crier()
def faire_crier(animal):
    print(animal.crier())


faire_crier(Canard())  # Coin coin !
faire_crier(Robot())   # Bip bip ! — le Robot « cancane », donc il passe

---

## Les interfaces en Python : deux approches

Python n'a pas de mot-clé `interface` comme Java ou C#, mais il offre **deux mécanismes** pour définir un contrat :

1. Les **classes abstraites** (`ABC` + `@abstractmethod`) : contrat par **héritage** — la classe doit explicitement hériter de l'interface (« il faut signer le contrat »).
2. **`typing.Protocol`** (depuis Python 3.8) : contrat par **forme** — il suffit que l'objet possède les bonnes méthodes, sans aucun héritage (« il suffit d'avoir les bonnes méthodes »). C'est le duck typing, mais vérifiable par les outils.

Commençons par l'approche classique avec `ABC`.

---

### Étape 1 : Définir l'interface (classe abstraite)

In [ ]:
from abc import ABC, abstractmethod

class Vehicule(ABC):
    @abstractmethod
    def accelerer(self, acceleration):
        pass

    @abstractmethod
    def freiner(self, deceleration):
        pass

---

### Étape 2 : Implémenter l'interface dans une classe concrète

La classe concrète **doit** fournir une implémentation de chaque méthode abstraite :

In [ ]:
# La classe concrète hérite de l'interface Vehicule définie à l'étape 1
# et implémente chacune de ses méthodes abstraites
class Voiture(Vehicule):
    def accelerer(self, acceleration):
        print(f"La voiture accélère de {acceleration} km/h")

    def freiner(self, deceleration):
        print(f"La voiture freine de {deceleration} km/h")


ma_voiture = Voiture()
ma_voiture.accelerer(30)
ma_voiture.freiner(10)

---

### Vérifier avec `isinstance()`

`isinstance(objet, Interface)` permet de vérifier si un objet implémente une interface :

In [ ]:
from abc import ABC, abstractmethod

class Interface(ABC):

    @abstractmethod
    def methode_interface(self):
        pass

class MaClasse(Interface):

    def methode_interface(self):
        print("Méthode de l'interface implémentée dans la classe")


# Attention : ne pas appeler la variable `object`, cela écraserait
# le type de base `object` de Python
mon_objet = MaClasse()

if isinstance(mon_objet, Interface):
    print("L'objet est une instance de l'interface")
else:
    print("L'objet n'est pas une instance de l'interface")

---

## `typing.Protocol` : le contrat par la forme (Python 3.8+)

Un **Protocol** décrit la **forme** attendue d'un objet : la liste des méthodes qu'il doit posséder. Contrairement à `ABC`, la classe qui respecte le contrat **n'a pas besoin d'hériter** du Protocol — il suffit qu'elle ait les bonnes méthodes. C'est le duck typing rendu officiel et vérifiable.

- Les outils d'analyse statique (mypy, Pyright, votre IDE) vérifient le contrat **sans exécuter le code**.
- Avec le décorateur `@runtime_checkable`, on peut aussi utiliser `isinstance()` à l'exécution.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Roulant(Protocol):
    """Contrat : tout objet qui sait rouler."""
    def rouler(self) -> str: ...


# Ces deux classes n'héritent de RIEN : elles ont juste la bonne méthode
class Velo:
    def rouler(self):
        return "Le vélo roule"

class Camion:
    def rouler(self):
        return "Le camion roule"

class Bateau:
    def naviguer(self):
        return "Le bateau navigue"


def demarrer(vehicule: Roulant):
    print(vehicule.rouler())


demarrer(Velo())    # OK : Velo a la méthode rouler()
demarrer(Camion())  # OK : Camion aussi

# Grâce à @runtime_checkable, isinstance vérifie la FORME de l'objet
print(isinstance(Velo(), Roulant))    # True  — sans aucun héritage !
print(isinstance(Bateau(), Roulant))  # False — pas de méthode rouler()

---

## ABC vs Protocol : lequel choisir ?

| | **ABC** (classe abstraite) | **Protocol** |
|---|---|---|
| Type de contrat | Par **héritage** : « il faut signer le contrat » | Par **forme** : « il suffit d'avoir les bonnes méthodes » |
| La classe doit hériter ? | Oui (`class Voiture(Vehicule)`) | Non — aucun lien requis |
| Vérification | À l'instanciation (`TypeError` si incomplet) | Par les outils de typage ; à l'exécution avec `@runtime_checkable` |
| Peut fournir du code partagé ? | Oui (méthodes concrètes héritées) | Non (uniquement la forme) |
| Cas d'usage typique | Hiérarchie de classes que vous contrôlez | Accepter des objets tiers sans les modifier |

> **En entretien** : « Python a-t-il des interfaces ? » — Pas de mot-clé `interface`, mais deux équivalents : les classes abstraites (`abc.ABC`, contrat imposé par héritage) et `typing.Protocol` depuis Python 3.8 (contrat structurel : seule la forme de l'objet compte, dans l'esprit du duck typing).